# GENIE BPC CRC: liver metastasis and early treatment failure, aggregate-only

This notebook is the Challenge 10 example: an analytic layer that returns only
aggregate tables, with disclosure control, so that its output can be shared with
colleagues who do not hold access to the patient-level GENIE BPC data.

It runs on the real CRC v2.0-public release when `GENIEKIT_DATA_DIR` points at a
folder that contains `CRC_v2.0-public` (downloaded with `geniekit download CRC`).
Otherwise it runs on a **synthetic** CRC cohort generated by `geniekit.synthetic`:
the tables then have the right structure, but every value is invented and must
not be reported as a finding.

What the report contains:

- (a) patients stratified by liver metastasis status: demographics, stage,
  institution, treatment received;
- (b) early treatment failure at 6 and 9 months after the start of first-line
  systemic therapy for advanced disease (progression, next line, or death);
- (c) frequencies of APC, TP53, KRAS, NRAS, BRAF V600E and SMAD4 alterations
  and of MSI-H or dMMR status;
- (d) comparisons of early failure between liver and non-liver metastasis and
  between genomic subgroups (RAS, BRAF V600E, MSI-H or dMMR).

In [ ]:
import os
from pathlib import Path

import pandas as pd

from geniekit import aggregate, io, lot, synthetic

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

data_dir = os.environ.get("GENIEKIT_DATA_DIR")
release = Path(data_dir) / "CRC_v2.0-public" if data_dir else None
if release is not None and release.exists():
    crc = io.load_release(release)
    source = "real release"
else:
    crc = synthetic.generate("CRC", n_patients=400, seed=1)
    source = "SYNTHETIC cohort: every value is invented"
print(crc)
print("Data source:", source)

## Lines of therapy

The early-failure endpoint needs the first line of systemic therapy for advanced
disease. The index event is the user's choice; here it is `"advanced"` (stage IV
at diagnosis, or the first distant metastasis), and the rules are the defaults
(`lot.LineRules()`), all of which are written into the output.

In [ ]:
lines = lot.derive(crc, index="advanced")
print(lines)
lines.summary()

## The report

`DisclosureControl(min_count=5)` suppresses any count below 5 and every statistic
derived from such a count; where a suppressed cell could be recovered from a
table margin, the layer either suppresses a second cell or omits the small
category, whichever hides fewer records, and says so in the table's notes.

In [ ]:
control = aggregate.DisclosureControl(min_count=5)
report = aggregate.liver_metastasis_report(crc, control=control, lines=lines)
print(report)
for note in report.notes:
    print("-", note)
report.summary()

### (a) Patients by liver metastasis status

In [ ]:
for key in ["patients_by_liver_mets", "liver_mets_timing", "liver_mets_by_stage", "sex_by_liver_mets", "age_by_liver_mets", "first_line_by_liver_mets", "lines_by_liver_mets"]:
    table = report[key]
    print(f"\n== {key}  (n = {table.n})")
    display(table.frame)
    for note in table.notes:
        print("  note:", note)

### (b) Early treatment failure after first-line therapy

For each horizon, `evaluable` records either failed on or before the horizon or
were followed beyond it; `failed %` is the proportion among them with a Wilson
interval, and `km failed %` the Kaplan-Meier cumulative failure probability over
all records that started a first line. The comparison column gives the risk
difference (percentage points) and a Fisher exact p-value against the reference
group.

In [ ]:
table = report["early_failure_by_liver_mets"]
display(table.frame.T)
for note in table.notes:
    print("  note:", note)
display(report["overall_survival_by_liver_mets"].frame)

### (c) Genomic alterations and MSI/MMR status

In [ ]:
display(report["alterations_by_liver_mets"].frame)
for note in report["alterations_by_liver_mets"].notes:
    print("  note:", note)
display(report["msi_mmr_by_liver_mets"].frame)

### (d) Early failure by genomic subgroup

In [ ]:
for key in ["early_failure_by_ras", "early_failure_by_braf_v600e", "early_failure_by_msi_mmr"]:
    if key in set(report):
        print(f"\n== {key}")
        display(report[key].frame.T)

## Sharing the output

`to_dir` writes one CSV per table and a `report.json` with the settings (index
event, line rules, disclosure control), the notes and every suppressed cell. The
folder contains aggregate tables only.

In [ ]:
folder = report.to_dir("outputs/crc_liver_metastasis")
sorted(p.name for p in folder.iterdir())

## Your own questions

`aggregate.analytic_table` builds the patient-level frame the report summarises
(it stays with you), and the summary functions turn any column of it into a
disclosure-controlled table:

In [ ]:
table = aggregate.analytic_table(crc, lines=lines, genes=["KRAS", "BRAF"], msi=True)
adv = table[table["index_days"].notna()]
aggregate.proportion_table(adv, "first_line", by="stage_dx", control=control).frame